# Dynamic Factor Model (Turkey, Pipeline B)

**Model**: DFM via `nowcastDFM::dfm`  
**Target**: `ngdprsaxdctrq`  
**Variables**: Complete Cat3 monthly set (22 predictors) + target = 23 total. COVID dummies and Tier-C short-history variables excluded.  
**Train**: 1995-01-01 to 2011-12-31 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: YES (factor loadings depend on variance; EM algorithm standardizes internally)  
**HP tuning**: NO (1 global + per-block factors per Bok et al. 2018)  
**Estimation**: EM algorithm, one fit (not rolling). Kalman filter handles ragged-edge nowcasting for the Cat3 monthly panel.  
**Prediction window**: 30-year rolling window (360 months) passed to predict_dfm (memory bound).  
**Deviation from US DFM**: Turkey has only 2 quarterly vars (ngdprsaxdctrq + real_gdp_i). US DFM excluded all  
quarterly Cat3 because nQ=23 caused eye((5*23)^2)=115x115 OOM. With nQ=2, eye(100) is trivial — both quarterly  
vars retained. real_gdp_i (a GDP-index series) provides additional Kalman signal for the common GDP factor.  
**COVID dummies excluded**: Zero variance in 1995-2011 training window crashes set_prior() / EM init.  
Same treatment as US DFM (Lenza-Primiceri 2022 defense: dummies used where variance is non-zero).

In [ ]:
library(tidyverse)
library(nowcastDFM)

source("../data/helpers.R")

metadata <- read_csv("../turkey_data/meta_data_tr.csv", show_col_types = FALSE)

# All Cat3 vars (22) — monthly only (COVID excluded: zero variance in training)
cat3_features <- c(
  "altin_rezerv_var", "auto_prod", "bist100", "consu_i", "cpi_sa",
  "deposit_i", "doviz_rezerv_var", "emp_rate", "empl_num", "fin_acc",
  "ipi_sa", "m3", "maden_ciro_endeksi_sa", "ppi", "reer",
  "resmi_rezerv_var", "tax", "total_prod", "tourist", "unemp_num",
  "unemp_rate", "usd_try_avg"
)

# All Tier-C vars (32) — short history, DFM-only; Kalman filter handles sparse starts
# real_gdp_i is quarterly (freq=q) — included because nQ=2 total (trivial Kalman size)
tier_c_features <- c(
  "1week-repo", "auto_exp_vol_i", "auto_imp_vol_i", "bus_conf",
  "card_pmt_i", "card_pmt_i_r", "card_trans", "commprop_sales_sa",
  "cons_conf", "cur_sa", "dg_exp_vol_i", "dg_imp_vol_i",
  "elec_cons", "elec_prod", "exp_ind_auto", "exp_ind_dg",
  "exp_vol_i", "hh_pmt", "hh_pmt_r", "house_sales_sa",
  "household_credits", "imp_ind_auto", "imp_ind_dg", "imp_vol_i",
  "is_ekon_ciro_sa", "loans", "non_fin_firms_credits", "real_gdp_i",
  "retail_sales_nsa", "retail_sales_sa", "sanayi_ins_tic_ciro_sa",
  "total_prop_sales"
)

dfm_features <- cat3_features

data <- read_csv("../turkey_data/data_tf_monthly_tr.csv", show_col_types = FALSE) %>%
    arrange(date)

target_variable  <- "ngdprsaxdctrq"
train_start_date <- "1995-01-01"
test_start_date  <- "2018-03-01"
test_end_date    <- "2025-12-01"

# End-of-quarter dates (Mar/Jun/Sep/Dec) for test period
test_dates <- seq(as.Date("2018-03-01"), as.Date(test_end_date), by = "3 months")

test <- data %>%
    dplyr::filter(date >= as.Date(train_start_date), date <= as.Date(test_end_date)) %>%
    data.frame()

for (col in colnames(test)) {
    if (is.numeric(test[, col]) && sum(is.infinite(test[, col])) > 0) {
        test[is.infinite(test[, col]), col] <- NA
    }
}

cat("DFM Turkey: target =", target_variable, "| Cat3 =", length(cat3_features),
    "| Tier-C available =", length(tier_c_features),
    "| Used =", length(dfm_features) + 1, "vars (Cat3 + target; Tier-C excluded)\n")

In [ ]:
# Patch nowcastDFM:::init_conds — cov(res[, idx_iM]) fails when a block has exactly
# 1 monthly variable because R drops dimensions, returning a vector instead of a matrix.
# Fix: wrap with as.matrix(..., drop=FALSE) to preserve 2-D shape.
# Turkey block_l has 4 vars, block_s has 4 — unlikely to trigger, included for safety.
local({
  fn <- nowcastDFM:::init_conds
  b  <- body(fn)
  replace_cov <- function(node) {
    if (is.call(node)) {
      txt <- deparse(node)
      if (identical(txt, "cov(res[, idx_iM])")) {
        return(quote(cov(as.matrix(res[, idx_iM, drop = FALSE]))))
      }
      return(as.call(lapply(node, replace_cov)))
    }
    node
  }
  body(fn) <- replace_cov(b)
  assignInNamespace("init_conds", fn, ns = "nowcastDFM")
  cat("nowcastDFM:::init_conds patched (drop=FALSE for single-variable blocks)\n")
})

# Patch dfm to cap A eigenvalues to prevent explosion with highly missing data
local({
  fn <- nowcastDFM::dfm; b <- body(fn)
  replace_A <- function(node) {
    if (is.call(node)) {
      txt <- deparse(node)[1]
      if (grepl("A <- em_output\\$A_new", txt)) {
        return(quote({
          A <- em_output$A_new
          diag_A <- diag(A)
          diag_A <- pmin(pmax(diag_A, -0.95), 0.95)
          diag(A) <- diag_A
        }))
      }
      return(as.call(lapply(node, replace_A)))
    }
    node
  }
  body(fn) <- replace_A(b)
  assignInNamespace("dfm", fn, ns="nowcastDFM")
})


# Build DFM variable list: preserves column order from data_tf_monthly_tr.csv
col_names_dfm <- colnames(test)[2:ncol(test)]
col_names_dfm <- col_names_dfm[col_names_dfm %in% c(target_variable, dfm_features)]
col_names_dfm <- c(col_names_dfm[col_names_dfm != target_variable], target_variable)

# Build blocks from metadata (Bok et al. 2018 4-block structure)
blocks <- metadata %>%
    dplyr::filter(series %in% col_names_dfm) %>%
    dplyr::filter(series %in% colnames(test))
blocks <- blocks[match(col_names_dfm, blocks$series), ]
blocks <- blocks %>%
    select(starts_with("block_")) %>%
    select_if(~ sum(.) > 0) %>%
    data.frame()
print(paste("Blocks:", ncol(blocks), "block columns for", nrow(blocks), "variables"))
print(paste("Variables matched:", length(col_names_dfm)))

# Fit DFM once on training data (1995-2011) — static model, not rolling
train_cols <- c("date", col_names_dfm)
train <- test %>%
    dplyr::filter(date <= as.Date("2011-12-31")) %>%
    dplyr::filter(date >= as.Date(train_start_date)) %>%
    data.frame()
train <- train[, train_cols]
train$date <- as.character(train$date)   # dfm() requires character dates
cat("Fitting DFM on", nrow(train), "months x", nrow(blocks), "variables...\n")
output_dfm <- dfm(data = train, blocks = blocks, max_iter = 500)
cat("DFM fitted\n")

In [ ]:
# Rolling prediction loop
vintage_offsets <- c(m1 = -2, m2 = -1, m3 = 0)
pred_dict <- data.frame(date = test_dates)
for (lag_name in names(vintage_offsets)) {
    pred_dict[, lag_name] <- NA
}

# Cap window to 30 years (360 months) to bound Kalman smoother memory.
# Turkey training starts 1995; test starts 2018 (~276 months). No trimming needed
# in early quarters, but included for robustness as test progresses toward 2025.
WINDOW_MONTHS <- 360

for (i in 1:length(test_dates)) {
    for (lag_name in names(vintage_offsets)) {
        vintage_date <- shift_month(test_dates[i], vintage_offsets[[lag_name]])
        lagged_data <- gen_vintage_data(metadata, test, test_dates[i], vintage_date)
        lagged_data <- data.frame(lagged_data)
        lagged_data <- lagged_data[, train_cols]  # match DFM training columns
        # Keep date as Date (NOT character): predict_dfm's add_month loop assigns as.Date()
        # back to the date column; if column is character, Date coerces to integer ("17258")
        # causing substr() to return "" -> month=NA -> if(month==12) crashes.
        lagged_data[lagged_data$date == test_dates[i], target_variable] <- NA

        # Trim to rolling window
        if (nrow(lagged_data) > WINDOW_MONTHS) {
            lagged_data <- tail(lagged_data, WINDOW_MONTHS)
        }

        prediction <- tryCatch({
            predict_dfm(lagged_data, output_dfm) %>%
                dplyr::filter(date == test_dates[i]) %>%
                select(!!target_variable) %>%
                pull()
        }, error = function(e) {
            cat("ERR", i, lag_name, conditionMessage(e), "\n")
            NA
        })
        pred_dict[pred_dict$date == test_dates[i], lag_name] <- prediction
    }
    if (i %% 8 == 0) print(paste(i, "/", length(test_dates)))
}

# Extract actuals at quarter-end months matching test_dates
actuals <- test %>%
    dplyr::filter(date %in% as.Date(test_dates)) %>%
    select(!!target_variable) %>%
    pull()

In [ ]:
# Save predictions
dir.create("../turkey_predictions", showWarnings = FALSE)
for (lag_name in names(vintage_offsets)) {
    df_out <- data.frame(
        date = test_dates,
        actual = actuals,
        prediction = pred_dict[, lag_name]
    )
    write.csv(df_out, paste0("../turkey_predictions/dfm_tr_", lag_name, ".csv"), row.names = FALSE)
    cat("Saved dfm_tr_", lag_name, ".csv (\n", nrow(df_out), " rows)\n", sep="")
}

# Evaluation
panels <- list(
    "Pre-crisis (2018-2019)" = c("2018-03-01", "2019-12-31"),
    "COVID      (2020-2021)" = c("2020-06-01", "2021-12-31"),
    "Post-COVID (2022-2025)" = c("2022-03-01", "2025-12-31"),
    "Full test  (2018-2025)" = c("2018-03-01", "2025-12-31")
)

rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm = TRUE))
mae  <- function(a, p) mean(abs(a - p), na.rm = TRUE)

cat("DFM (Turkey) — Evaluation by Panel & Vintage\n")
for (pn in names(panels)) {
    rng <- panels[[pn]]
    m <- test_dates >= rng[1] & test_dates <= rng[2]
    cat("--- ", pn, " ---\n")
    for (lag_name in names(vintage_offsets)) {
        cat("  ", lag_name, "  RMSFE=",
            format(rmse(actuals[m], pred_dict[m, lag_name]), digits = 6),
            "  MAE=",
            format(mae(actuals[m], pred_dict[m, lag_name]), digits = 6), "\n")
    }
}